In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define paths to the data files
data_folder = '/content/drive/MyDrive/datasets/manipuri MT/English-Manipuri/parallel'
train_src_file = os.path.join(data_folder, 'en-mni-train-mni.txt')
train_tgt_file = os.path.join(data_folder, 'en-mni-train-en.txt')
valid_src_file = os.path.join(data_folder, 'en-mni-valid-mni.txt')
valid_tgt_file = os.path.join(data_folder, 'en-mni-valid-en.txt')
test_src_file = os.path.join(data_folder, 'en-mni-test-mni.txt')
test_tgt_file = os.path.join(data_folder, 'en-mni-test-en.txt')

In [ ]:
# Load sentences
def load_sentences(src_file, tgt_file):
    src_sentences = []
    tgt_sentences = []
    with open(src_file, 'r', encoding='utf-8') as src_f, open(tgt_file, 'r', encoding='utf-8') as tgt_f:
        for src_line, tgt_line in zip(src_f, tgt_f):
            src_sentences.append(src_line.strip())
            tgt_sentences.append(tgt_line.strip())
    return src_sentences, tgt_sentences

train_src, train_tgt = load_sentences(train_src_file, train_tgt_file)
valid_src, valid_tgt = load_sentences(valid_src_file, valid_tgt_file)
test_src, test_tgt = load_sentences(test_src_file, test_tgt_file)

print("Sample Training Source Sentences:", train_src[:4])
print("Sample Training Target Sentences:", train_tgt[:4])

Sample Training Source Sentences: ['এনর্জি পোর্টেল অসিনা মথক্কী ৱারোলশীং অসিগী ঈ-পাউ অসি নহাক্না মখোয়বু শীজিন্ননিংগদবা অমদি মখোয়গা লোয়নরিবা কান্নবশীং লৌবা পামহন্নবগী মওংদা পুক্নিং থৌগৎনিংঙাই ওইবা ৱারিশীংগা লোয়ননা পীনবা হোৎনৈ ।', 'মহৌশাগী উপদ্রবশীংনা শোকহল্লবা লৌমীশীংদা রিলিফ পীনবগীদমক্তা ,  বেঙ্কশীংদা অহানবা চহিদুগী ওইনা রিষ্ট্রকচর তৌরবা এমাউন্ট অদুদা ২%গী ইন্টরেষ্ট সবভেন্সন পীগনি ।', 'প্রধানমন্ত্রী শ্রী নরেন্দ্র মোদীনা লুচিংবা কেন্দ্রগী মন্ত্রীমন্দলনা দিপার্তমেন্ত ওফ ইকোনোমীক এফিয়ার্স ( ইন্দিয়ান ইকোনোমীক সর্বিস কেদর )  অমসুং দি ত্রেজরী , গবর্নমেন্ত ওফ ওস্ত্রেলিয়াগী মরক্তা থা অহুমগী ওইনা সেকেন্দমেন্ত প্রোগ্রামগীদমক মেমোরেন্দা ওফ অন্দর্সতেন্দিং খুৎয়েক পিন্নবা য়াখ্রে ।', 'মসিনা অঙাং উনবা মতমদা প্রোব্লেম থোকহনবা য়াই ,  অমসুং অঙাং অমদি মমাদা ত্রোমা  লৈহনবা ,  অমসুং পোক্লবা মতুংদা অঙাং অদুগী ব্লদ গ্লুকোজদু খংহৌদনা লেপখিবনচিংবগী থৌওংশীং থোকহনবা য়াই ।']
Sample Training Target Sentences: ['the energy portal attempts to give information on the above aspects with inspiring stories that would mot

In [ ]:
# Tokenize and pad sequences
class Vocab:
    def __init__(self, sentences):
        self.word2index = {}
        self.index2word = {}
        self.word2count = {}
        self.n_words = 0
        self.build_vocab(sentences)

    def build_vocab(self, sentences):
        for sentence in sentences:
            for word in sentence.split():
                self.add_word(word)
        # Add a special token for unknown words
        self.add_word('<UNK>')

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.word2count[word] = 1
            self.n_words += 1
        else:
            self.word2count[word] += 1

    def sentence_to_indices(self, sentence):
        return [self.word2index.get(word, self.word2index['<UNK>']) for word in sentence.split()] # Handle unknown words

    def indices_to_sentence(self, indices):
        return ' '.join([self.index2word[idx] for idx in indices])

src_vocab = Vocab(train_src)
tgt_vocab = Vocab(train_tgt)

def tokenize_and_pad(sentences, vocab, max_len):
    tokenized = [vocab.sentence_to_indices(sentence) for sentence in sentences]
    padded = pad_sequence([torch.tensor(seq) for seq in tokenized], batch_first=True, padding_value=0)
    return padded

max_len = max(max(len(sent.split()) for sent in train_src), max(len(sent.split()) for sent in train_tgt))

train_src_indices = tokenize_and_pad(train_src, src_vocab, max_len)
train_tgt_indices = tokenize_and_pad(train_tgt, tgt_vocab, max_len)
valid_src_indices = tokenize_and_pad(valid_src, src_vocab, max_len)
valid_tgt_indices = tokenize_and_pad(valid_tgt, tgt_vocab, max_len)
test_src_indices = tokenize_and_pad(test_src, src_vocab, max_len)
test_tgt_indices = tokenize_and_pad(test_tgt, tgt_vocab, max_len)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, src_data, tgt_data):
        self.src_data = src_data
        self.tgt_data = tgt_data

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]

train_dataset = TranslationDataset(train_src_indices, train_tgt_indices)
valid_dataset = TranslationDataset(valid_src_indices, valid_tgt_indices)
test_dataset = TranslationDataset(test_src_indices, test_tgt_indices)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
#define device if available cuda , else cpu
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cpu


In [ ]:
#!export CUDA_LAUNCH_BLOCKING=1
#!export TORCH_USE_CUDA_DSA=1

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, tgt, hidden, cell):
        embedded = self.embedding(tgt)
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        predictions = self.fc_out(outputs)
        return predictions, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        hidden, cell = self.encoder(src)
        output, hidden, cell = self.decoder(tgt, hidden, cell)
        return output

input_dim = src_vocab.n_words
output_dim = tgt_vocab.n_words
emb_dim = 256
hidden_dim = 512
n_layers = 2

encoder = Encoder(input_dim, emb_dim, hidden_dim, n_layers)
decoder = Decoder(output_dim, emb_dim, hidden_dim, n_layers)
model = Seq2Seq(encoder, decoder)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
from tqdm import tqdm

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for i, (src, tgt) in enumerate(tqdm(iterator, desc="Training", leave=False)):
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt)
        output_dim = output.shape[-1]
        output = output.view(-1, output_dim)
        tgt = tgt.view(-1)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, tgt in tqdm(iterator, desc="Evaluating", leave=False):
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt)
            output_dim = output.shape[-1]
            output = output.view(-1, output_dim)
            tgt = tgt.view(-1)
            loss = criterion(output, tgt)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)

n_epochs = 10
clip = 1

for epoch in range(n_epochs):
    print(f'Epoch {epoch+1}/{n_epochs}')
    train_loss = train(model, train_loader, optimizer, criterion, clip)
    valid_loss = evaluate(model, valid_loader, criterion)
    print(f'Train Loss: {train_loss:.3f}, Val. Loss: {valid_loss:.3f}')


Epoch 1/10


Train Loss: 3.526, Val. Loss: 1.517
Epoch 2/10


Train Loss: 0.837, Val. Loss: 0.716
Epoch 3/10


Train Loss: 0.242, Val. Loss: 0.502
Epoch 4/10


Train Loss: 0.047, Val. Loss: 0.458
Epoch 5/10


...

Train Loss: 0.003, Val. Loss: 0.459
Epoch 7/10


Train Loss: 0.002, Val. Loss: 0.459
Epoch 8/10


Train Loss: 0.001, Val. Loss: 0.461
Epoch 9/10


Train Loss: 0.001, Val. Loss: 0.463
Epoch 10/10


Train Loss: 0.000, Val. Loss: 0.463


In [ ]:
# Save the model
torch.save(model.state_dict(), 'LSTM_model2.pt')

# Load the model (for inference)
model.load_state_dict(torch.load('LSTM_model2.pt'))

<All keys matched successfully>

In [ ]:
# Load the model (for inference)
try:
    model.load_state_dict(torch.load('/content/drive/MyDrive/Manipuri MT/mt-LSTM.pt', map_location=torch.device('cpu')))
    print("Model loaded successfully from LSTM_model2.pt")
except FileNotFoundError:
    print("Error: Model file not found. Check the file path.")
except RuntimeError as e:
    print(f"Error loading model: {e}")

Error loading model: Error(s) in loading state_dict for Seq2Seq:
	Missing key(s) in state_dict: "encoder.lstm.weight_ih_l0", "encoder.lstm.weight_hh_l0", "encoder.lstm.bias_ih_l0", "encoder.lstm.bias_hh_l0", "encoder.lstm.weight_ih_l1", "encoder.lstm.weight_hh_l1", "encoder.lstm.bias_ih_l1", "encoder.lstm.bias_hh_l1", "decoder.lstm.weight_ih_l0", "decoder.lstm.weight_hh_l0", "decoder.lstm.bias_ih_l0", "decoder.lstm.bias_hh_l0", "decoder.lstm.weight_ih_l1", "decoder.lstm.weight_hh_l1", "decoder.lstm.bias_ih_l1", "decoder.lstm.bias_hh_l1". 
	Unexpected key(s) in state_dict: "encoder.rnn.weight_ih_l0", "encoder.rnn.weight_hh_l0", "encoder.rnn.bias_ih_l0", "encoder.rnn.bias_hh_l0", "encoder.rnn.weight_ih_l1", "encoder.rnn.weight_hh_l1", "encoder.rnn.bias_ih_l1", "encoder.rnn.bias_hh_l1", "decoder.rnn.weight_ih_l0", "decoder.rnn.weight_hh_l0", "decoder.rnn.bias_ih_l0", "decoder.rnn.bias_hh_l0", "decoder.rnn.weight_ih_l1", "decoder.rnn.weight_hh_l1", "decoder.rnn.bias_ih_l1", "decoder.rnn.bi

In [ ]:
# prompt: test the model by using random sentences from the test data and then compare with original test target data

import random

# Choose random indices from the test dataset
random_indices = random.sample(range(len(test_src_indices)), 5)

# Get the corresponding source and target sentences
test_src_sentences = [test_src[i] for i in random_indices]
test_tgt_sentences = [test_tgt[i] for i in random_indices]

# Tokenize and pad the source sentences
test_src_indices = tokenize_and_pad(test_src_sentences, src_vocab, max_len)

# Tokenize and pad the target sentences as well
test_tgt_indices = tokenize_and_pad(test_tgt_sentences, tgt_vocab, max_len) # Add this line

# Move data to the device
test_src_indices = test_src_indices # Move to device
test_tgt_indices = test_tgt_indices # Move to device

# Generate predictions
model.eval()
with torch.no_grad():
    predictions = model(test_src_indices, test_tgt_indices) # Pass in target indices

# Convert predictions to sentences
predicted_sentences = []
for prediction in predictions:
    predicted_indices = prediction.argmax(dim=-1).tolist() # Convert to list
    # Filter out indices that are not in the vocabulary and padding tokens (index 0)
    valid_indices = [idx for idx in predicted_indices if idx in tgt_vocab.index2word and idx != 0]
    predicted_sentences.append(tgt_vocab.indices_to_sentence(valid_indices))

# Print the results
for i in range(len(random_indices)):
    print(f"Source Sentence: {test_src_sentences[i]}")
    print(f"Original Target Sentence: {test_tgt_sentences[i]}")
    print(f"Predicted Target Sentence: {predicted_sentences[i]}")
    print()

Source Sentence: মহাক্না ঐগী একাউন্টশীংদা মেসেজ লেপ্তনা থাজিলক্লে
Original Target Sentence: he keeps sending me messages on my different accounts
Predicted Target Sentence: shiny sneeze accompanying accompanying Pharmaceuticals backend Students repeal repeal much apartments Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri

Source Sentence: ষ্ট্রকচর  ,   ওদিএন্স স্পেসিফিক এপ্রোচকী খুত্থাংদা কম্যুনিটি এৱারনেস হেন্না লৈহন্বা
Original Target Sentence: raising community awareness through structured , audience specific approach
Predicted Target Sentence: shiny Baat ,636 Dhanamanjuri cisterns Dhanamanjuri facilitated Turnbull signedin Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanjuri Dhanamanju